# Dia 2 - Disponibilização de dados para BI e Analytics
- Objetivos: apresentar como disponibilizar dados limpos e modelos para consumo de ferramentas como PowerBI, Tableu, Metabase e para uso em aprendizado de máquina. Além disso, tem como objetivo apresentar os conceitos de reverse ETL e governança de dados.

(Ana Julia e Luiza)
- Consumo por BI vs Ad-hoc
    - explorar as diferenças entre requisitoos recorrentes corporativos e análises exploratórias pontuais
- criação de visões analíticas no dw: CREATE VIEW e CREATE MATERIALIZED VIEW no duckDB para expor visões simplificadas da tabela fato e dimensões sem duplicar dados
https://www.w3schools.com/sql/sql_view.asp


- camada semântica / metrics layer
    - discutir sobre padronização de métricas de negócio em um único local
https://www.ibm.com/br-pt/think/topics/semantic-layer

- apresentar uma demonstração de utitlização das viewas criadas a partir de dashboards criados com pandas+matplotlib/plotly/seaborn


(Francisco e Fabrícia)
- disponibilização para ciência de dados e ML (feature store)
    - consumo para ML
        - ex: agregação histórica de comportamento de compra de clientes
    - feature store
        - https://www.databricks.com/br/blog/what-is-a-feature-store
    - exportação otimizada
        - https://www.datacamp.com/pt/tutorial/data-preparation-with-pandas
        - exportar como parquet e loading com pandas aplicando para um modelo do scikit-learn

- reverse elt
    - analytics passivo vs analytics operacional
    https://hightouch.com/blog/reverse-etl
    - paradigma reverse etl: dw como fonte para outras aplicações
    - discutir sobre boas práticas e cuidados ao elaborar os ciclos de auto alimentação com apis e outros serviços
    https://realpython.com/python-requests/


(daivik)
- governança, qualidade e contratos do serving
    - data trust
    - sla (service level agreement) e slo (service level objective)
        - prazos de entrega
        - metas de disponibilidade
        - pipelines de dados
    - data contracts

    - entrega: 
        - elaborar uma demonstração de como um data mart pode ser validado a partir do python, considerando métricas de nulidade ou limites aceitáveis
        - gerar casos de uso (sucesso e falha), emitindo um alerta de qualidade de dados em caso de falha na validação
    https://www.ibm.com/br-pt/think/topics/data-contract

## Governança, qualidade e contratos do serving

> **Ideia central:** dado só tem valor se quem usa acredita nele.

**Exemplo:** uma rede de lojas tem um dashboard de vendas que a diretoria abre toda segunda de manhã. Numa segunda, o dashboard mostra que as vendas da região Sudeste **caíram 40%**. A diretoria entra em pânico e cogita cortar o orçamento de marketing da região. Duas horas depois, alguém descobre que as vendas não caíram: uma das lojas passou a enviar a região como **"SE"** em vez de **"Sudeste"**, e o dashboard não contou essas vendas. O erro foi corrigido no mesmo dia, mas na segunda seguinte o diretor pergunta: *"Esse número está certo mesmo? Confirma com a planilha do financeiro antes."*

O maior estrago não foi o número errado, que foi corrigido rápido, e sim a **perda de confiança**. Confiança demora muito para ser construída e um único erro basta para perdê-la.

Até aqui vimos como **expor** os dados do DW: views para BI, métricas padronizadas, features para ML e reverse ETL. Falta uma pergunta importante: **o consumidor pode confiar nesse dado?**

Um dashboard bonito com números errados é pior do que não ter dashboard, porque decisões serão tomadas em cima dele. É isso que chamamos de **data trust** (confiança nos dados): a garantia de que o dado entregue está **correto, completo, atualizado e no formato combinado**.

A confiança depende de algumas dimensões de qualidade:

| Dimensão | Pergunta que responde | Exemplo de falha |
|---|---|---|
| **Completude** | Tem dado faltando? | `ticket_medio` nulo em várias linhas |
| **Validade** | Os valores fazem sentido? | Receita negativa |
| **Consistência** | Segue o padrão combinado? | Região `"SE"` em vez de `"Sudeste"` |
| **Unicidade** | Tem registro duplicado? | Mesma região e dia aparecendo duas vezes |
| **Atualidade** (*freshness*) | O dado chegou no prazo? | Dashboard de hoje mostrando dados de 4 dias atrás |
| **Volume** | Chegou a quantidade esperada? | Carga com 10 linhas quando o normal é 150 |

Nesta seção vamos:
1. Entender **SLA e SLO**, a forma de combinar prazos e metas com quem consome os dados.
2. Conhecer **data contracts**, o acordo formal entre quem produz e quem consome um dado.
3. Construir em Python um **validador de data mart** baseado em contrato, com um caso de **sucesso** e um caso de **falha** que emite um **alerta de qualidade**.

### SLA e SLO

> **Ideia central:** SLA e SLO são a forma de transformar "o dado vai chegar logo" em uma promessa concreta e mensurável.

**Exemplo:** a diretora comercial pergunta ao time de dados: *"A que horas o dashboard de vendas fica pronto?"*. O time responde: *"Ah, de manhã cedo, geralmente."* Num dia, o dashboard só atualiza às 11h, e a diretora já tinha feito a reunião das 9h com números de dois dias atrás. Ela reclama, e o time responde: *"Mas a gente disse 'geralmente'..."*. Ninguém mentiu, mas também **ninguém combinou nada**: "de manhã cedo, geralmente" não dá para cobrar nem medir.

Esses termos vêm da engenharia de software (SRE) e foram adotados pela engenharia de dados:

| Sigla | Nome | O que é | Exemplo |
|---|---|---|---|
| **SLA** | *Service Level Agreement* | O **acordo** com o consumidor, com consequências se for quebrado | Dados de D-1 prontos até as 07:00 em **90%** dos dias; abaixo disso o time abre um incidente e comunica a diretoria |
| **SLO** | *Service Level Objective* | A **meta interna** do time | Dados de D-1 prontos até as 07:00 em **95%** dos dias |

Repare que o **SLO é mais rígido que o SLA**. Essa diferença funciona como margem de segurança: se o SLO começa a ser quebrado, o time percebe e corrige **antes** de violar o acordo com o cliente.

Para saber se as metas estão sendo cumpridas, o time **mede** o resultado real (por exemplo: "no último mês, o dado ficou pronto no prazo em 27 de 30 dias"). Esse número medido é chamado de **SLI** (*Service Level Indicator*).

Em dados, os SLAs normalmente cobrem três aspectos:
- **Prazos de entrega**: "os dados de ontem (D-1) estarão disponíveis até as 07:00".
- **Metas de disponibilidade**: "o DW estará acessível para consulta 99,5% do tempo no mês".
- **Pipelines de dados**: "pelo menos 98% das execuções diárias terminam com sucesso".

Ferramentas de orquestração como Airflow, Dagster e Prefect permitem configurar prazos e alertas para acompanhar essas metas de forma automática.

### Data contracts

> **Ideia central:** um data contract é um acordo escrito entre quem produz um dado e quem usa, dizendo exatamente o que o dado vai ter e quais garantias ele cumpre.

**Exemplo:** o time de engenharia de dados acha que o nome da coluna `receita_total` está confuso e a renomeia para `faturamento`. Tudo roda sem erro. No dia seguinte, o **dashboard comercial** mostra um campo vazio, o **modelo de previsão de vendas** do time de ciência de dados quebra e o **relatório do financeiro** para de atualizar. Ninguém agiu de má-fé: a engenharia nem sabia que esses três times usavam aquela coluna, porque **não havia nada escrito** dizendo isso.

Um **data contract** (contrato de dados) é um acordo **explícito e versionado** entre o **produtor** de um dado (o time de engenharia que mantém o data mart) e seus **consumidores** (dashboards, analistas, modelos de ML, sistemas via reverse ETL).

Sem contrato, cada consumidor "descobre" o formato do dado na prática, e qualquer mudança (renomear uma coluna, mudar um tipo) quebra algo em silêncio. Com contrato, o formato e as garantias ficam escritos e podem ser **verificados automaticamente**.

Um contrato costuma conter:
- **Identificação**: nome, versão, dono (quem acionar se der problema) e consumidores.
- **Schema**: colunas e tipos esperados.
- **Regras de qualidade**: limites de nulos, faixas de valores aceitáveis, valores permitidos, chaves únicas.
- **SLA**: atualidade máxima, volume mínimo e prazos de entrega.

Na prática, contratos são escritos em YAML ou JSON (há inclusive um padrão aberto, o *Open Data Contract Standard*) e verificados com ferramentas como **dbt tests**, **Great Expectations**, **Soda** ou **Pandera**. Aqui vamos representar o contrato como um dicionário Python e escrever o validador do zero, para entender o que essas ferramentas fazem por baixo dos panos.

### Demonstração: validando um data mart com Python

Agora vamos colocar em prática o que vimos: pegar uma tabela do data mart, conferir se ela respeita as regras combinadas no contrato e **emitir um alerta** caso alguma regra seja quebrada.

A demonstração tem quatro passos:
1. **A tabela do data mart**: a tabela que será entregue aos consumidores.
2. **As regras do contrato**: o que combinamos que o dado deve garantir.
3. **A validação**: o código que confere cada regra.
4. **O alerta**: o aviso emitido quando algo está errado.

Depois, testamos dois casos: uma tabela **correta** (sucesso) e uma tabela **com erros** (falha).

#### 1. A tabela do data mart

Este é o data mart de **Vendas**, usado pelo time comercial. Uma empresa costuma ter vários data marts (Financeiro, RH...), e **cada um tem o próprio contrato** com regras diferentes. Aqui vamos validar apenas o de Vendas.

No DW, este data mart é modelado em esquema estrela: uma tabela fato de vendas e dimensões como loja (que guarda a região de cada loja) e data. Além dessas tabelas, o data mart costuma ter **tabelas agregadas**: resumos já calculados, para que o dashboard não precise somar milhões de vendas a cada acesso.

A tabela abaixo é uma dessas tabelas agregadas. Ela é o resultado de uma **consulta** que junta cada venda com a sua loja (JOIN) e soma as vendas por região (GROUP BY). O pipeline roda essa consulta a cada carga e salva o resultado no data mart.

Um data mart, no geral, é validado **tabela por tabela**, cada uma com as próprias regras. Aqui mostramos o processo com esta tabela agregada, que é a que o dashboard recebe. A tabela fato e as dimensões seguiriam o mesmo processo.

No fluxo completo do curso, essa consulta seria feita no DW (por exemplo, no DuckDB). Aqui, para simplificar, vamos criar a tabela final diretamente com o pandas.

In [ ]:
import pandas as pd

data_mart = pd.DataFrame({
    "regiao":        ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"],
    "qtd_pedidos":   [120, 135, 110, 150, 140],
    "receita_total": [30000.0, 34000.0, 27500.0, 37000.0, 35000.0],
    "ticket_medio":  [250.0, 251.9, 250.0, 246.7, 250.0],
})

data_mart

#### 2. As regras do contrato

Este dicionário é uma versão simplificada do **data contract**: ele guarda as regras que combinamos com os consumidores do data mart.
- **Nulidade**: cada coluna pode ter no máximo 5% de valores vazios.
- **Limites aceitáveis**: a receita deve estar entre 0 e 1 milhão, e o ticket médio entre 1 e 5 mil.

In [ ]:
regras = {
    "max_nulos": 0.05,  # cada coluna pode ter no máximo 5% de valores vazios
    "limites": {
        "receita_total": (0, 1_000_000),
        "ticket_medio": (1, 5_000),
    },
}

#### 3. A validação

A função `validar` confere cada regra e devolve uma **lista de problemas** encontrados. Se a lista estiver vazia, o data mart passou em todas as regras.

In [ ]:
def validar(df, regras):
    problemas = []

    # Regra 1: nulidade
    for coluna in df.columns:
        pct_nulos = df[coluna].isna().mean()  # percentual de valores vazios na coluna
        if pct_nulos > regras["max_nulos"]:
            problemas.append(f"{coluna}: {pct_nulos:.0%} de valores vazios")

    # Regra 2: limites aceitáveis
    for coluna, (minimo, maximo) in regras["limites"].items():
        fora = df[(df[coluna] < minimo) | (df[coluna] > maximo)]  # linhas fora do intervalo
        if len(fora) > 0:
            problemas.append(f"{coluna}: {len(fora)} valor(es) fora do intervalo {minimo} a {maximo}")

    return problemas

#### 4. O alerta

A função `verificar_data_mart` roda a validação e decide: se não houver problemas, o data mart é aprovado; se houver, emite um **alerta de qualidade de dados** listando o que deu errado.

Aqui o alerta é apenas impresso na tela, mas em uma empresa ele seria enviado para o Slack, Teams ou e-mail do time responsável.

In [ ]:
def verificar_data_mart(df, regras):
    problemas = validar(df, regras)
    if not problemas:
        print("[OK] Data mart aprovado. Pode ser disponibilizado.")
    else:
        print("[ALERTA DE QUALIDADE DE DADOS] Data mart reprovado!")
        for problema in problemas:
            print("  -", problema)

#### Caso 1: data mart correto (sucesso)

O data mart que criamos respeita todas as regras, então deve ser aprovado.

In [ ]:
verificar_data_mart(data_mart, regras)

#### Caso 2: data mart com erros (falha)

Agora vamos simular dois problemas comuns na vida real:
- um **valor vazio** no ticket médio (uma integração deixou de enviar o campo);
- uma **receita negativa** (um bug no cálculo).

Fazemos uma cópia do data mart e "estragamos" esses valores.

In [ ]:
data_mart_com_erro = data_mart.copy()
data_mart_com_erro.loc[0, "ticket_medio"] = None       # valor vazio
data_mart_com_erro.loc[1, "receita_total"] = -5000.0   # receita negativa

verificar_data_mart(data_mart_com_erro, regras)

Repare que o alerta diz **exatamente o que está errado e em qual coluna**. Assim, o time responsável consegue investigar rápido, e o problema é descoberto **antes** de o dado chegar ao dashboard, e não pela diretoria.

Na vida real, as validações costumam incluir também outras regras, como duplicatas, valores permitidos e se o dado chegou no prazo. Ferramentas como **dbt tests**, **Great Expectations**, **Soda** e **Pandera** automatizam esse tipo de verificação.